# NOAA OLR Wheeler-Kiladis Spectra

This notebook recreates the symmetric and antisymmetric NOAA OLR spectra from the reference project. It is written as a gallery example: each processing step uses the public `xr_ccew` API so the figure-making workflow is visible and reusable.

The reference notebook produced two PDFs:

- `NOAA_OLR_symmetric_spectra.pdf`
- `NOAA_OLR_antisymmetric_spectra.pdf`

The full calculation uses the 1979-01-01 through 1996-08-31 NOAA 2x daily OLR record. For a fast smoke test, set `XR_CCEW_GALLERY_MAX_SEGMENTS=1` before executing the notebook.

In [ ]:
from __future__ import annotations

import os
from pathlib import Path

import numpy as np
import xarray as xr

import xr_ccew as tw
from xr_ccew.matsuno import (
    eig0_frequency,
    eig_frequency,
    er_frequency,
    kelvin_frequency,
    mrg_frequency,
    wig_frequency,
)

repo_root = Path.cwd()
if not (repo_root / "pyproject.toml").exists():
    repo_root = repo_root.parent

data_path = repo_root / "OLD_REFERENCE_PROJECT" / "data" / "olr.2xdaily.1979-2022.nc"
output_dir = Path(os.environ.get("XR_CCEW_GALLERY_OUTPUT_DIR", repo_root / "gallery" / "figures")).resolve()
output_dir.mkdir(parents=True, exist_ok=True)

os.environ.setdefault("MPLCONFIGDIR", str(output_dir / ".matplotlib"))
os.environ.setdefault("XDG_CACHE_HOME", str(output_dir / ".cache"))
Path(os.environ["MPLCONFIGDIR"]).mkdir(parents=True, exist_ok=True)
Path(os.environ["XDG_CACHE_HOME"]).mkdir(parents=True, exist_ok=True)

import matplotlib.colors as colors
import matplotlib.pyplot as plt

max_segments_env = os.environ.get("XR_CCEW_GALLERY_MAX_SEGMENTS")
max_segments = int(max_segments_env) if max_segments_env else None

time_start = "1979-01-01"
time_end = "1996-08-31"
lat_bound = 20.0
segment_days = 96
overlap_days = 30

print(f"input: {data_path}")
print(f"output: {output_dir}")
print(f"max_segments: {max_segments}")

## Load The Reference OLR Field

The old reference project used tropical NOAA OLR on a latitude-longitude-time grid. The latitude mask below is order-independent, so the notebook works whether the source latitude coordinate is ascending or descending.

In [ ]:
olr = xr.open_dataarray(data_path)
olr = olr.sel(time=slice(time_start, time_end))
olr = olr.where(np.abs(olr["lat"]) <= lat_bound, drop=True).sortby("lat")
olr.name = "olr"
olr

## Split Into Equatorial Symmetry Components

`xr_ccew.symmetric_antisymmetric_component` mirrors the field across the equator using interpolation. That means the decomposition does not require exact `+lat` and `-lat` grid pairs.

In [ ]:
olr_symmetric = tw.symmetric_antisymmetric_component(olr, "symmetric")
olr_antisymmetric = tw.symmetric_antisymmetric_component(olr, "antisymmetric")

olr_symmetric

## Remove Low-Frequency Background Structure

The reference workflow removes the mean and linear trend, then removes the first three harmonics of the seasonal cycle. These are public preprocessing functions so users can compose or replace this part of the pipeline.

In [ ]:
def remove_reference_background(field: xr.DataArray) -> xr.DataArray:
    detrended = tw.remove_mean_and_linear_trend(field)
    return tw.remove_harmonics_of_seasonal_cycle(detrended, num_harmonics=3)


symmetric_anomalies = remove_reference_background(olr_symmetric)
antisymmetric_anomalies = remove_reference_background(olr_antisymmetric)

symmetric_anomalies

## Segment, Window, Transform, And Average

For each 96-day segment with 30 days of overlap, the workflow applies a Tukey window, computes the space-time power spectrum, averages over segments, and sums over latitude.

In [ ]:
def segmented_power(field: xr.DataArray, *, name: str) -> xr.DataArray:
    segments = tw.segment_data(field, segment_days=segment_days, overlap_days=overlap_days)
    if max_segments is not None:
        segments = segments[:max_segments]
    if not segments:
        raise ValueError("No full segments are available for this date range.")

    print(f"{name}: {len(segments)} segment(s)")
    powers = []
    for segment in segments:
        windowed = tw.apply_window(segment, dim="time")
        powers.append(tw.power_spectrum(windowed))

    power = xr.concat(powers, dim="segment").mean("segment").sum("lat")
    power.name = name
    return power


symmetric_power = segmented_power(symmetric_anomalies, name="symmetric_power")
antisymmetric_power = segmented_power(antisymmetric_anomalies, name="antisymmetric_power")

symmetric_power

## Smooth Background And Ratio Spectra

The Wheeler-Kiladis-style display emphasizes power relative to a smoothed background. Here the background is the repeated 1-2-1 smoothing of the average of symmetric and antisymmetric power.

In [ ]:
background = tw.background_spectrum((symmetric_power + antisymmetric_power) / 2.0)

symmetric_ratio = symmetric_power / background
antisymmetric_ratio = antisymmetric_power / background

symmetric_ratio

## Matsuno Dispersion Curves

The shaded fields come from the data pipeline above. The curve overlays use the public Matsuno helpers to draw the same reference equivalent-depth families.

In [ ]:
def _add_matsuno_families(ax, *, he_list, families, color="k", linewidth=1.5):
    k = np.linspace(-50.0, 50.0, 500)
    for equivalent_depth in he_list:
        for make_frequency, linestyle in families:
            frequency = make_frequency(k, equivalent_depth)
            finite = np.isfinite(k) & np.isfinite(frequency)
            if finite.sum() >= 2:
                ax.plot(
                    k[finite],
                    frequency[finite],
                    color=color,
                    linestyle=linestyle,
                    linewidth=linewidth,
                )
    return ax


def add_symmetric_matsuno_modes(ax, *, he_list):
    families = [
        (lambda k, he: kelvin_frequency(k, he), "-"),
        (lambda k, he: eig_frequency(k, he, n=1), "--"),
        (lambda k, he: wig_frequency(k, he, n=1), ":"),
        (lambda k, he: er_frequency(k, he, n=1), "-."),
    ]
    return _add_matsuno_families(ax, he_list=he_list, families=families)


def add_antisymmetric_matsuno_modes(ax, *, he_list):
    families = [
        (lambda k, he: mrg_frequency(k, he), "-"),
        (lambda k, he: eig0_frequency(k, he), "-"),
        (lambda k, he: eig_frequency(k, he, n=2), "--"),
        (lambda k, he: wig_frequency(k, he, n=2), ":"),
    ]
    return _add_matsuno_families(ax, he_list=he_list, families=families)

## Plot And Save The Figure Family

Both figures use the same contour levels and axes as the old reference output. The only difference is that the data processing above is now expressed through `xr_ccew`.

In [ ]:
def plot_ratio_spectrum(ratio: xr.DataArray, title: str, *, mode: str):
    if mode not in {"symmetric", "antisymmetric"}:
        raise ValueError("mode must be 'symmetric' or 'antisymmetric'")

    fig, ax = plt.subplots(figsize=(4, 4))
    lev_min = 1.1
    lev_max = 1.6
    interval = 0.1
    levels = np.arange(lev_min, lev_max + interval, interval)

    ratio.plot.contourf(
        ax=ax,
        levels=levels,
        cmap=plt.cm.Blues,
        norm=colors.Normalize(vmin=lev_min, vmax=lev_max),
        extend="max",
    )

    ax.set_ylim(0, 0.5)
    ax.set_xlim(-15, 15)
    ax.set_xlabel("k_zonal")
    ax.set_ylabel("CPD")
    ax.set_title(title)

    if mode == "symmetric":
        add_symmetric_matsuno_modes(ax, he_list=[8, 12, 25, 50])
    else:
        add_antisymmetric_matsuno_modes(ax, he_list=[8, 12, 25, 50])

    return fig, ax


symmetric_fig, symmetric_ax = plot_ratio_spectrum(symmetric_ratio, "NOAA OLR", mode="symmetric")
symmetric_path = output_dir / "NOAA_OLR_symmetric_spectra.pdf"
symmetric_fig.savefig(symmetric_path, format="pdf", bbox_inches="tight")
symmetric_path

In [ ]:
antisymmetric_fig, antisymmetric_ax = plot_ratio_spectrum(
    antisymmetric_ratio,
    "NOAA OLR",
    mode="antisymmetric",
)
antisymmetric_path = output_dir / "NOAA_OLR_antisymmetric_spectra.pdf"
antisymmetric_fig.savefig(antisymmetric_path, format="pdf", bbox_inches="tight")
antisymmetric_path

The regenerated PDFs are now in `gallery/figures/` unless `XR_CCEW_GALLERY_OUTPUT_DIR` was set before execution.